In [ ]:
## Notebook 01 — Data Extraction & Target Variable Construction
**Input:** Raw LendingClub CSV (accepted_2007_to_2018Q4.csv) — 2,260,701 rows × 150+ columns  
**Output:** accepted_columns_curtail.csv — 2,260,701 rows × 28 columns  
**Purpose:** Select origination-time features only and construct binary default target.

### Key Decisions
- 27 features selected: only variables observable at loan origination retained.  
  Post-origination fields excluded to prevent target leakage.
- default_flag = 1 for: Charged Off, Default. default_flag = 0 for: Fully Paid.  
  Ambiguous statuses (Current, Late, In Grace Period) set to NULL and excluded from modeling population.
- 'Does not meet the credit policy. Status: Charged Off' (n=760, 0.03%) excluded from mapping.  
  Materiality assessment confirmed <1bp impact on default rate. Documented transparently.
- Note on tooling: MySQL extraction returned incomplete results on the 2.26M row dataset.  
  Extraction performed directly in Python using pandas usecols for memory efficiency.

In [1]:
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine
print('done')

done


In [5]:
file_path = "/Users/abhinavsaxena/Documents/Project/1/Raw_Data/accepted_2007_to_2018Q4.csv"
output_path = "/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_columns_curtail.csv"

keep_cols = [
    'id','issue_d','term','loan_amnt','funded_amnt','int_rate','installment',
    'grade','sub_grade','emp_length','home_ownership','annual_inc',
    'verification_status','purpose','addr_state','dti','delinq_2yrs',
    'earliest_cr_line','fico_range_low','fico_range_high','open_acc',
    'total_acc','revol_util','inq_last_6mths','pub_rec','application_type',
    'loan_status'
]

print("Loading only selected columns...")
df = pd.read_csv(file_path, usecols=keep_cols, low_memory=False)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")

def map_default_flag(status):
    if pd.isna(status):
        return None
    status = status.strip()
    if status in ['Charged Off', 'Default']:
        return 1
    elif status == 'Fully Paid':
        return 0
    else:
        return None  # for Current, Late, etc.

df['default_flag'] = df['loan_status'].apply(map_default_flag)

print("default_flag added successfully")

print(f"Saving to {output_path} ...")

if output_path.endswith(".csv"):
    df.to_csv(output_path, index=False)
else:
    df.to_parquet(output_path, index=False)

print("Done! File saved successfully with default_flag added.")


Loading only selected columns...
Loaded 2,260,701 rows and 27 columns
default_flag added successfully
Saving to /Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_columns_curtail.csv ...
Done! File saved successfully with default_flag added.


In [7]:
import pandas as pd

# Load the cleaned dataset 
df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_columns_curtail.csv")

# View first few rows
print("Preview of cleaned dataset:")
display(df.head())

# See columns
print("\nColumns in dataset:")
print(df.columns.tolist())

# See basic info
print("\nData types and non-null counts:")
print(df.info())



/var/folders/lh/cdggvq1555ncf31qp3kpmrmm0000gn/T/ipykernel_1741/2141527006.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/abhinavsaxena/Documents/Project/1/clean_data/accepted_columns_curtail.csv")


Preview of cleaned dataset:


,id,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_util,total_acc,application_type,default_flag
0,68407277,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,...,Aug-2003,675.0,679.0,1.0,7.0,0.0,29.7,13.0,Individual,0.0
1,68355089,24700.0,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,...,Dec-1999,715.0,719.0,4.0,22.0,0.0,19.2,38.0,Individual,0.0
2,68341763,20000.0,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,...,Aug-2000,695.0,699.0,0.0,6.0,0.0,56.2,18.0,Joint App,0.0
3,66310712,35000.0,35000.0,60 months,14.85,829.90,C,C5,10+ years,MORTGAGE,...,Sep-2008,785.0,789.0,0.0,13.0,0.0,11.6,17.0,Individual,NaN
4,68476807,10400.0,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,...,Jun-1998,695.0,699.0,3.0,12.0,0.0,64.5,35.0,Individual,0.0



Columns in dataset:
['id', 'loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_util', 'total_acc', 'application_type', 'default_flag']

Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 28 columns):
 #   Column               Dtype  
---  ------               -----  
 0   id                   object 
 1   loan_amnt            float64
 2   funded_amnt          float64
 3   term                 object 
 4   int_rate             float64
 5   installment          float64
 6   grade                object 
 7   sub_grade            object 
 8   emp_length           object 
 9   home_ownership       object 
 10  annual_inc           floa

In [8]:
memory_gb = df.memory_usage(deep=True).sum() / 1e9
print(f"\n💾 Memory used by this DataFrame: {memory_gb:.2f} GB")


💾 Memory used by this DataFrame: 1.89 GB
